# Hedge experiments

Two hedge ideas tested against / alongside the chento Triple sleeve. Re-runnable viewer; engines are the `.py` files in this folder. Full prod-faithful methodology (build_optimized + replay_with_mae + apply_filters) — nothing here touches production.

- **Test 1 — reactive hedge:** enter single-direction; if price goes the wrong side of entry, open an opposite leg instead of stopping, unwind on a bounce, then ride the original leg. (`reactive_hedge.py`)
- **Test 2 — range-formation hedge:** when a range forms, long the range low + short the range high; if the range breaks, the favorable-extreme leg still wins. Standalone strategy. (`range_hedge.py`)

Prior context: B13 (`validation_B13_hedge*.py`) already showed hedging *at the Triple trigger* is decisively negative; these test two *different* hedge mechanics.

In [ ]:
from pathlib import Path
import json, pandas as pd
from IPython.display import Image, display
D = Path.cwd()
show = lambda f: display(Image(filename=str(D / f))) if (D / f).exists() else print('missing (run the engine):', f)
def jshow(f):
    p = D / f
    print(p.read_text() if p.exists() else f'run the engine first -> {f}')

## Test 1 — reactive hedge (vs single-direction, prod exit model)

Sweeps `hedge_at_R` (adverse level that triggers the hedge, inside the 5×ATR stop) × `unwind_R` (bounce that closes it). Baseline = current single-direction sleeve. Regenerate: `python reactive_hedge.py`.

**Result:** roughly neutral — best cells ≈ baseline MAR, most worsen drawdown. Did NOT cap DD (baseline already strings almost no losses).

In [ ]:
jshow('reactive_hedge_results.txt')
show('reactive_hedge_equity.png')

## Test 2 — range-formation hedge (standalone)

Long limit at range low, short limit at range high, TP at 90% across the range, breakout stop = 1×ATR beyond the edge, 18bp/leg cost. Evaluated as its own strategy (separate sleeve candidate per B13). Regenerate: `python range_hedge.py`.

**Result:** decisively negative — backward range detection makes you fade breakouts (fill the extreme leg exactly as the range breaks).

In [ ]:
jshow('range_hedge_results.txt')
show('range_hedge_equity.png')

## Conclusions (2026-06-09)

**Neither hedge mechanic improves the sleeve — do not pursue either as a change.**

**Test 1 — reactive hedge:** roughly neutral. Baseline MAR 92.6 (maxDD −2.4R). The 9 hedge@X/unwindY cells are noisy with no monotonic pattern; the best two (hedge@0.5) reach MAR ~95 with the same −2.4R drawdown (marginal, likely noise at n≈100), while most cells WORSEN drawdown (−3.1 to −3.9R) and MAR (57–78). It did NOT cap drawdown — the 72%-WR baseline strings almost no losses, so there's little tail to insure, and the unwind whipsaw can deepen DD. As predicted: hedge-and-hold ≡ a tighter stop; hedge-and-unwind leans on the anti-edge opposite leg.

**Test 2 — range hedge:** decisively negative (meanR −1.34, WR 9%, MAR −1.0; only 1/380 events had both legs win). Backward range detection makes you fade breakouts. Needs a forward-confirming, oscillation-counting detector (a separate, bigger build).

**Net:** consistent with B13 — hedging doesn't help the Triple. The only hedge-adjacent path with any promise is a *dedicated* range-fishing sleeve built on a much better range detector; deferred.

Decision gate (per workflow rules): nothing here clears the bar to touch production.